<a href="https://colab.research.google.com/github/Dines-pro/Generative-AI-document-chatbot/blob/main/Codepart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Streamlit and Pyngrok
!pip install streamlit pyngrok
!pip install langchain
!pip install -U langchain-community
!pip install pypdf
!pip install sentence-transformers
!pip install chromadb

In [ ]:
%%writefile app.py

import os
import streamlit as st
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.llms import HuggingFacePipeline

# Define the document loader and text splitter
loader = PyPDFLoader("/content/INDIAN_PENAL_CODE.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
texts = text_splitter.split_documents(documents)


In [ ]:
# Set up the embeddings and vector database
persist_directory = "/content/vectordb"
os.makedirs(persist_directory, exist_ok=True)
embeddings = SentenceTransformerEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")
db = Chroma.from_documents(texts, embeddings, persist_directory=persist_directory)
db.persist()

# Load or initialize language model for text generation
checkpoint = "MBZUAI/LaMini-Flan-T5-783M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
base_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, device_map="auto", torch_dtype=torch.float32)
pipe = pipeline("text2text-generation", model=base_model, tokenizer=tokenizer, max_length=512, do_sample=True, temperature=0.3, top_p=0.95)

# Initialize QA chain
local_llm = HuggingFacePipeline(pipeline=pipe)
qa_chain = RetrievalQA.from_chain_type(llm=local_llm, chain_type="stuff", retriever=db.as_retriever(search_type="similarity", search_kwargs={"k": 2}), return_source_documents=True)

# Streamlit app
st.title("NITHIBOT ⚖️")
st.write("Hello! I am a legal chat bot specializing in Indian Penal Code queries. I'm here to help answer any questions you have regarding the Indian Penal Code.")

# Input field for user query
input_query = st.text_input("Enter your query:")

# Execute the query when user submits
if input_query:
    llm_response = qa_chain({"query": input_query})
    response = llm_response['result']
    st.subheader("Response:")
    st.write(response)

    # Display source document metadata if available
    #st.subheader("Source Document(s):")
    #for doc in llm_response["source_documents"]:
        #st.write(doc.metadata)

In [ ]:
!rm -rf ~/.ngrok2/ngrok.yml


In [ ]:
# Run Streamlit in the background
!streamlit run app.py &>/dev/null &

# Import ngrok and start a single tunnel
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(f"Access your Streamlit app here: {public_url}")
